# leafmap: análisis geoespacial y mapeo interactivo

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gf0657-programacionsig/2026-ii/blob/main/contenidos/iv-procesamiento-datos-geoespaciales/21-leafmap-mapeo-interactivo.ipynb)

## Trabajo previo

### Lecturas

Antes de la clase, revise las siguientes lecturas, en inglés. La guía de inicio de la documentación oficial muestra, en una página, cómo se crea un mapa de leafmap y qué hace su barra de herramientas; el artículo de Wu presenta la biblioteca, sus objetivos y sus bloques de construcción, y es breve.

Wu, Q. (s. f.). Get started. En *leafmap documentation*. Recuperado el 25 de octubre de 2026, de https://leafmap.org/get-started/
\
\
Wu, Q. (2021). Leafmap: A Python package for interactive mapping and geospatial analysis with minimal coding in a Jupyter environment. *Journal of Open Source Software, 6*(63), 3414. [https://joss.theoj.org/papers/10.21105/joss.03414](https://joss.theoj.org/papers/10.21105%2Fjoss.03414)

### Otros recursos

La [galería de cuadernos de leafmap](https://leafmap.org/notebooks/00_key_features/) tiene un ejemplo ejecutable por cada funcionalidad, y la [referencia de la API](https://leafmap.org/leafmap/) documenta cada método de la clase `Map` y cada función del módulo. La [documentación de ipyleaflet](https://ipyleaflet.readthedocs.io/), la biblioteca de *widgets* sobre la que leafmap construye sus mapas, explica los controles y las capas que leafmap envuelve; la de [localtileserver](https://localtileserver.banesullivan.com/) describe cómo se sirven los rasters locales al mapa.

## Introducción

El cuaderno anterior construyó mapas web con folium pieza por pieza: el resultado es una página HTML que se publica, pero la comunicación va en un solo sentido, de Python al navegador. [leafmap](https://leafmap.org/), creada por Qiusheng Wu, de la Universidad de Tennessee, es una biblioteca de código abierto para **mapeo interactivo y análisis geoespacial con el mínimo de código** en cuadernos de Jupyter. Se construye sobre [ipyleaflet](https://ipyleaflet.readthedocs.io/), que hace del mapa un **widget** de Jupyter: un objeto que vive a la vez en Python y en el navegador, de modo que lo que el usuario hace en el mapa (dibujar un polígono, mover un control deslizante, hacer clic) vuelve a Python como datos. Sobre ese mapa, leafmap agrega una barra de herramientas, más de 150 mapas base, métodos que cargan capas vectoriales y raster en una línea, un servidor de teselas para rasters locales, comparación de capas y funciones de análisis y conversión de datos.

La relación con folium es de complemento, no de sustitución. folium produce páginas que se publican; leafmap produce mapas con los que se trabaja mientras se analiza, y además puede usar folium como motor (`leafmap.foliumap`) cuando lo que se quiere es publicar. Las dos comparten Leaflet, los mapas base y buena parte del vocabulario. Este cuaderno recorre leafmap con los mismos datos del anterior (las capas del IGN, la población del INEC, la altitud de WorldClim) y con la imagen Sentinel-2 del refugio Gandoca-Manzanillo del cuaderno de rasterio, que es donde leafmap aporta más: rasters en un mapa web sin preprocesarlos.

Una limitación práctica: los *widgets* no sobreviven a la conversión a HTML estático. Por eso, en la versión publicada en el sitio web del curso, cada mapa de este cuaderno aparece como una **captura de pantalla** (una figura); al ejecutar el cuaderno en Colab o en Visual Studio Code, los mapas son interactivos. Como todos los cuadernos del curso, puede ejecutarse en la nube, con la insignia "Abrir en Google Colab", o localmente, con el ambiente `geopython` como kernel, que incluye leafmap y localtileserver. Colab no trae leafmap: la primera celda lo instala y activa el administrador de *widgets* cuando detecta que corre allí. El archivo `.ipynb` puede descargarse con el botón de descarga (ícono de flecha hacia abajo) en la parte superior de la página del cuaderno, en el sitio web del curso.

In [1]:
# Instalación y configuración solo en Google Colab (en un ambiente local, esta celda no hace nada)
try:
    import google.colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    %pip install -q leafmap localtileserver
    from google.colab import output
    output.enable_custom_widget_manager()

In [2]:
import os

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
import ipyleaflet
import leafmap

# Direcciones de los directorios de datos en el repositorio del curso
DATOS_IGN = "https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/datos/ign"
DATOS_INEC = "https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/datos/inec"
DATOS_WC = "https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/datos/worldclim"
DATOS_S2 = "https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/datos/sentinel"
DATOS_NE = "https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/datos/natural-earth"

# Colores de los cuadernos anteriores
AZUL = "#1F77B4"
NARANJA = "#D97B1B"

print("leafmap", leafmap.__version__)
print("ipyleaflet", ipyleaflet.__version__)

leafmap 0.63.0
ipyleaflet 0.20.0


## Carga de las capas

Las capas vectoriales se leen con geopandas, como siempre. A diferencia de folium, leafmap **reproyecta a WGS84 por su cuenta** las capas que recibe en otro SRC, de modo que las capas del IGN en CRTM05 (EPSG:8908) pueden agregarse tal cual; se reproyectan aquí de todos modos, para extraer las coordenadas de los aeródromos como columnas, que es lo que varios métodos de leafmap esperan. La población del INEC se une por el código del cantón:

In [3]:
# Capas del IGN y población del INEC, como en el cuaderno de folium
provincias = gpd.read_file(f"{DATOS_IGN}/provincias.gpkg").to_crs(4326)
cantones = gpd.read_file(f"{DATOS_IGN}/cantones.gpkg").to_crs(4326)
aerodromos = gpd.read_file(f"{DATOS_IGN}/aerodromos.gpkg").to_crs(4326)

poblacion = pd.read_csv(f"{DATOS_INEC}/cantones-2022.csv")
cantones = cantones.merge(poblacion[["codigo_canton", "poblacion", "area_km2", "densidad"]], on="codigo_canton", how="left")

# Coordenadas de los aeródromos como columnas, para los métodos que reciben tablas
aerodromos["longitud"] = aerodromos.geometry.x
aerodromos["latitud"] = aerodromos.geometry.y

print(len(provincias), "provincias |", len(cantones), "cantones |", len(aerodromos), "aeródromos")
aerodromos.head(3)

7 provincias | 84 cantones | 119 aeródromos


,nombre,codigo_distrito,geometry,longitud,latitud
0,Aeródromo Barra de Tortuguero,70206,POINT (-83.51538 10.56979),-83.515381,10.569792
1,Aeródromo Barra de Colorado,70206,POINT (-83.58564 10.76907),-83.585641,10.769070
2,Campo de Aterrizaje,60702,POINT (-83.35476 8.40191),-83.354758,8.401912


Los rasters se sirven al mapa desde archivos, por lo que conviene tenerlos en el directorio de trabajo. `leafmap.download_file()` descarga un archivo de una URL (y no lo vuelve a descargar si ya existe, salvo con `overwrite=True`): la altitud de Centroamérica de WorldClim y la imagen Sentinel-2 de Gandoca-Manzanillo del cuaderno de rasterio (11 MB):

In [4]:
# Descarga de los rasters al directorio de trabajo
ALTITUD = leafmap.download_file(f"{DATOS_WC}/altitud-centroamerica.tif", "altitud-centroamerica.tif", quiet=True)
GANDOCA = leafmap.download_file(f"{DATOS_S2}/gandoca-20240204.tif", "gandoca-20240204.tif", quiet=True)

for archivo in [ALTITUD, GANDOCA]:
    print(os.path.basename(archivo), f"{os.path.getsize(archivo) / 1e6:.1f} MB")

altitud-centroamerica.tif 0.1 MB
gandoca-20240204.tif 11.4 MB


## Un primer mapa

`leafmap.Map()` crea el mapa. Sus argumentos se parecen a los de folium, con dos diferencias de nombre: `center` es una tupla `(latitud, longitud)` y `zoom` es el nivel de acercamiento inicial; `height` fija la altura. El mapa trae de serie el mapa base de OpenStreetMap, los botones de acercamiento y pantalla completa, las **herramientas de dibujo** (línea, polígono, rectángulo, círculo y marcador, en la columna izquierda) y, en la esquina superior derecha, el ícono de llave que despliega la **barra de herramientas**: desde ahí se cambia el mapa base, se abren el control de capas y el **inspector** (que muestra los valores de las capas al hacer clic), se agregan datos y se miden distancias, todo sin escribir código:

In [5]:
# Un mapa de leafmap: center es (latitud, longitud)
CENTRO_CR = (9.75, -84.15)

mapa = leafmap.Map(center=CENTRO_CR, zoom=8, height="500px")
mapa

<figure style="text-align: center;">
  <img src="https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/contenidos/iv-procesamiento-datos-geoespaciales/img/leafmap-01-primer-mapa.png" alt="Mapa de leafmap centrado en Costa Rica con las herramientas de dibujo y la barra de herramientas" width="900">
  <figcaption><strong>Figura 1</strong>. Un mapa de leafmap recién creado: a la izquierda, los controles de acercamiento, pantalla completa y dibujo; arriba a la derecha, el ícono de la barra de herramientas. Fuente: elaboración propia.</figcaption>
</figure>

Los controles se activan o desactivan al crear el mapa (`draw_control`, `measure_control`, `fullscreen_control`, `scale_control`, `toolbar_control`, entre otros). El método `add_basemap()` agrega mapas base por su nombre; el catálogo `leafmap.basemaps` tiene más de 150, los mismos de xyzservices que usó folium más algunos propios, y el control de capas (ícono de capas, o la barra de herramientas) permite elegir entre ellos:

In [6]:
# Controles elegidos y dos mapas base adicionales
mapa = leafmap.Map(center=CENTRO_CR, zoom=8, height="500px", draw_control=False, measure_control=True, scale_control=True)

mapa.add_basemap("OpenTopoMap")
mapa.add_basemap("Esri.WorldImagery")

print(len(leafmap.basemaps), "mapas base disponibles; por ejemplo:", list(leafmap.basemaps)[:4])
mapa

155 mapas base disponibles; por ejemplo: ['OpenStreetMap', 'Esri.WorldStreetMap', 'Esri.WorldImagery', 'Esri.WorldTopoMap']


<figure style="text-align: center;">
  <img src="https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/contenidos/iv-procesamiento-datos-geoespaciales/img/leafmap-02-mapas-base.png" alt="Mapa con el mapa base de imágenes de satélite de Esri y el control de capas abierto" width="900">
  <figcaption><strong>Figura 2</strong>. Mapas base agregados con add_basemap(): el control de capas (arriba a la derecha) permite alternarlos; el último agregado es el visible. Fuente: elaboración propia.</figcaption>
</figure>

*Ejercicios de esta sección: 1, en la sección de ejercicios al final del cuaderno.*

## Capas vectoriales

`add_gdf()` agrega un GeoDataFrame como capa, con `layer_name` para el control de capas, `style` y `hover_style` (diccionarios con los nombres de estilo de Leaflet, como en folium) e `info_mode`, que decide si los atributos se muestran al pasar el cursor (`"on_hover"`) o al hacer clic (`"on_click"`), en un recuadro en la esquina inferior derecha. `zoom_to_layer` centra el mapa en la capa; con `False` se conserva el centro indicado. Los cantones:

In [7]:
# Polígonos con estilo, resaltado y atributos al pasar el cursor
mapa = leafmap.Map(center=CENTRO_CR, zoom=8, height="500px")

mapa.add_gdf(
    cantones,
    layer_name="Cantones",
    style={"color": AZUL, "weight": 1, "fillColor": AZUL, "fillOpacity": 0.1},
    hover_style={"fillOpacity": 0.5},
    info_mode="on_hover",
    zoom_to_layer=False
)
mapa

<figure style="text-align: center;">
  <img src="https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/contenidos/iv-procesamiento-datos-geoespaciales/img/leafmap-03-cantones.png" alt="Los 84 cantones de Costa Rica sobre OpenStreetMap, con un cantón resaltado y sus atributos en un recuadro" width="900">
  <figcaption><strong>Figura 3</strong>. Una capa vectorial agregada con add_gdf(): el estilo, el resaltado y el recuadro de atributos. Fuente: elaboración propia.</figcaption>
</figure>

`add_data()` hace el mapa de coropletas: recibe la capa, la columna, el esquema de clasificación (`scheme`, con los nombres de mapclassify: `"Quantiles"`, `"NaturalBreaks"`, `"EqualInterval"`…), el número de clases `k`, la paleta `cmap` y el título de la leyenda, que construye sola con los intervalos de cada clase. La densidad de población por quintiles:

In [8]:
# Mapa de coropletas clasificado, con leyenda automática
mapa = leafmap.Map(center=CENTRO_CR, zoom=8, height="500px")

mapa.add_data(
    cantones,
    column="densidad",
    scheme="Quantiles",
    k=5,
    cmap="YlOrRd",
    legend_title="Densidad de población, 2022 (hab/km²)",
    layer_name="Densidad de población",
    style={"color": "white", "weight": 0.7, "fillOpacity": 0.8},
    info_mode="on_hover"
)
mapa

<figure style="text-align: center;">
  <img src="https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/contenidos/iv-procesamiento-datos-geoespaciales/img/leafmap-04-coropletas.png" alt="Mapa de coropletas de la densidad de población por cantón en cinco quintiles, con leyenda de intervalos" width="900">
  <figcaption><strong>Figura 4</strong>. Coropletas con add_data(): clasificación por quintiles y leyenda con los intervalos de cada clase. Fuente: elaboración propia.</figcaption>
</figure>

Monteverde y Puerto Jiménez, sin datos, reciben un color aparte. Las capas de puntos se agregan desde una tabla con columnas de coordenadas: `add_points_from_xy()` crea marcadores **agrupados** (con `popup` para las columnas de la ventana emergente) y `add_circle_markers_from_xy()`, círculos de radio fijo con estilo. Los aeródromos, de las dos maneras:

In [9]:
# Marcadores agrupados a partir de las columnas de coordenadas
mapa = leafmap.Map(center=CENTRO_CR, zoom=8, height="500px")

mapa.add_points_from_xy(
    aerodromos.drop(columns="geometry"),
    x="longitud", y="latitud",
    popup=["nombre", "codigo_distrito"],
    layer_name="Aeródromos"
)
mapa

<figure style="text-align: center;">
  <img src="https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/contenidos/iv-procesamiento-datos-geoespaciales/img/leafmap-05-marcadores.png" alt="Marcadores de los aeródromos, agrupados por cercanía" width="900">
  <figcaption><strong>Figura 5</strong>. Marcadores agrupados con add_points_from_xy(); al acercarse, los grupos se separan en marcadores individuales. Fuente: elaboración propia.</figcaption>
</figure>

In [10]:
# Círculos de radio fijo con estilo propio
mapa = leafmap.Map(center=CENTRO_CR, zoom=8, height="500px")

mapa.add_circle_markers_from_xy(
    aerodromos.drop(columns="geometry"),
    x="longitud", y="latitud",
    radius=5, color=NARANJA, fill_color=NARANJA, fill_opacity=0.8,
    popup=["nombre"],
    layer_name="Aeródromos"
)
mapa

<figure style="text-align: center;">
  <img src="https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/contenidos/iv-procesamiento-datos-geoespaciales/img/leafmap-06-circulos.png" alt="Los aeródromos como círculos naranja de radio fijo sobre el mapa de Costa Rica" width="900">
  <figcaption><strong>Figura 6</strong>. Marcadores circulares con add_circle_markers_from_xy(). Fuente: elaboración propia.</figcaption>
</figure>

`add_heatmap()` dibuja un mapa de calor de una tabla con coordenadas y una columna de valor (aquí, 1 para cada aeródromo, de modo que el mapa muestra la densidad de puntos). Y `add_wms_layer()` agrega una capa de un servicio WMS, como el del SNIT, con la misma dirección y los mismos nombres de capa que en folium (y la misma escala mínima: la red de drenaje se dibuja desde el nivel 12):

In [11]:
# Mapa de calor de los aeródromos
mapa = leafmap.Map(center=CENTRO_CR, zoom=8, height="500px")
mapa.add_basemap("Esri.WorldGrayCanvas")

mapa.add_heatmap(
    aerodromos.drop(columns="geometry").assign(peso=1),
    latitude="latitud", longitude="longitud", value="peso",
    name="Densidad de aeródromos", radius=20
)
mapa

<figure style="text-align: center;">
  <img src="https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/contenidos/iv-procesamiento-datos-geoespaciales/img/leafmap-07-calor.png" alt="Mapa de calor de los aeródromos sobre un fondo gris" width="900">
  <figcaption><strong>Figura 7</strong>. Mapa de calor con add_heatmap(): la concentración de aeródromos en Guanacaste, la península de Osa y el Caribe norte. Fuente: elaboración propia.</figcaption>
</figure>

In [12]:
# Capas del servicio WMS del IGN (SNIT) a escala 1:200 000
WMS_IGN_200 = "https://geos.snitcr.go.cr/be/IGN_200/wms?"

mapa = leafmap.Map(center=(9.93, -84.08), zoom=12, height="500px")
mapa.add_basemap("Esri.WorldGrayCanvas")

mapa.add_wms_layer(url=WMS_IGN_200, layers="reddrenaje_200k", name="Red de drenaje 1:200 000 (IGN)",
                   format="image/png", transparent=True, attribution="Instituto Geográfico Nacional, SNIT")
mapa.add_wms_layer(url=WMS_IGN_200, layers="redvial_200k", name="Red vial 1:200 000 (IGN)",
                   format="image/png", transparent=True, attribution="Instituto Geográfico Nacional, SNIT")
mapa

<figure style="text-align: center;">
  <img src="https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/contenidos/iv-procesamiento-datos-geoespaciales/img/leafmap-08-wms.png" alt="Red de drenaje en azul y red vial en rojo y negro del IGN sobre San José" width="900">
  <figcaption><strong>Figura 8</strong>. Capas WMS del SNIT agregadas con add_wms_layer(). Fuente: elaboración propia.</figcaption>
</figure>

*Ejercicios de esta sección: 2, 3, 4 y 5, en la sección de ejercicios al final del cuaderno.*

## Capas raster

Aquí está la mayor diferencia con folium. `add_raster()` recibe la ruta de un GeoTIFF local (o una URL) y lo muestra como una capa de teselas: por detrás, [localtileserver](https://localtileserver.banesullivan.com/) arranca un pequeño servidor que lee el archivo con rasterio, lo reproyecta y produce las teselas que el mapa pide, en el momento. No hay que convertir el raster a imagen ni reproyectarlo: `colormap`, `vmin` y `vmax` fijan la paleta y el rango, `nodata` el valor sin dato (si el archivo no lo declara) y `opacity` la transparencia. La altitud de Centroamérica, con una barra de color agregada con `add_colorbar()`:

In [13]:
# Raster local servido como teselas, con paleta, rango y barra de color
mapa = leafmap.Map(center=CENTRO_CR, zoom=7, height="500px")

mapa.add_raster(ALTITUD, colormap="terrain", vmin=0, vmax=3500, opacity=0.8, layer_name="Altitud (WorldClim)", zoom_to_layer=False)

paleta = matplotlib.colormaps["terrain"]
mapa.add_colorbar(colors=[matplotlib.colors.to_hex(paleta(v)) for v in np.linspace(0, 1, 8)],
                  vmin=0, vmax=3500, caption="Altitud (m)")
mapa

<figure style="text-align: center;">
  <img src="https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/contenidos/iv-procesamiento-datos-geoespaciales/img/leafmap-09-altitud.png" alt="Altitud de Centroamérica con la paleta terrain sobre el mapa base y una barra de color" width="900">
  <figcaption><strong>Figura 9</strong>. Un raster local agregado con add_raster(): las teselas las produce localtileserver a partir del GeoTIFF. Fuente: elaboración propia.</figcaption>
</figure>

Con una imagen multibanda, `indexes` elige las bandas que van a los canales rojo, verde y azul, y `vmin` y `vmax` hacen el **estiramiento** que en el cuaderno de rasterio se hizo con percentiles. Las bandas de la imagen Sentinel-2 son azul, verde, roja e infrarroja cercana (1 a 4), con números digitales entre unos 1 000 y 4 000 en las superficies terrestres; el color verdadero es `[3, 2, 1]`:

In [14]:
# Imagen Sentinel-2 en color verdadero: bandas roja, verde y azul, con estiramiento
mapa = leafmap.Map(center=(9.59, -82.63), zoom=13, height="500px")
mapa.add_basemap("Esri.WorldImagery")

mapa.add_raster(GANDOCA, indexes=[3, 2, 1], vmin=1000, vmax=3500, layer_name="Sentinel-2, 2024-02-04", zoom_to_layer=True)
mapa

<figure style="text-align: center;">
  <img src="https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/contenidos/iv-procesamiento-datos-geoespaciales/img/leafmap-10-sentinel.png" alt="Imagen Sentinel-2 del refugio Gandoca-Manzanillo en color verdadero sobre imágenes de Esri" width="900">
  <figcaption><strong>Figura 10</strong>. Una imagen multibanda con add_raster(): indexes elige las bandas y vmin y vmax el estiramiento. Fuente: elaboración propia.</figcaption>
</figure>

Para comparar dos capas, `split_map()` divide el mapa con un control deslizante: a la izquierda una capa, a la derecha otra, sobre el mismo fondo. Las capas pueden ser mapas base (por su nombre) o rasters (por su ruta, con los mismos argumentos de `add_raster()` en `left_args` y `right_args`). El color verdadero y el **falso color** (infrarroja cercana, roja y verde, con la vegetación en rojo) de la misma imagen:

In [15]:
# Comparación con control deslizante: color verdadero a la izquierda, falso color a la derecha
mapa = leafmap.Map(center=(9.59, -82.63), zoom=13, height="500px")

mapa.split_map(
    left_layer=GANDOCA, right_layer=GANDOCA,
    left_args={"indexes": [3, 2, 1], "vmin": 1000, "vmax": 3500, "layer_name": "Color verdadero"},
    right_args={"indexes": [4, 3, 2], "vmin": 1000, "vmax": 5000, "layer_name": "Falso color"},
    left_label="Color verdadero", right_label="Falso color"
)
mapa

<figure style="text-align: center;">
  <img src="https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/contenidos/iv-procesamiento-datos-geoespaciales/img/leafmap-11-comparacion.png" alt="Mapa dividido con la imagen Sentinel-2 en color verdadero a la izquierda y en falso color a la derecha" width="900">
  <figcaption><strong>Figura 11</strong>. Comparación de dos composiciones de la misma imagen con split_map(); el control deslizante se arrastra con el ratón. Fuente: elaboración propia.</figcaption>
</figure>

### Rasters en la nube

Un raster no tiene que estar en la computadora. Un **GeoTIFF optimizado para la nube** (COG), mencionado en el cuaderno de rasterio, está organizado en bloques y con vistas previas, de modo que un servidor puede leer de él solo la parte que necesita. `add_cog_layer()` recibe la URL de un raster en Internet y lo muestra con un servicio de teselas público ([titiler](https://developmentseed.org/titiler/)), que lee el archivo por partes: es la forma de ver en un mapa un raster publicado en un repositorio, sin descargarlo. La imagen de Gandoca, leída directamente del repositorio del curso:

In [16]:
# Raster remoto como capa de teselas: titiler lee el archivo del repositorio por partes
mapa = leafmap.Map(center=(9.59, -82.63), zoom=13, height="500px")

mapa.add_cog_layer(f"{DATOS_S2}/gandoca-20240204.tif", name="Sentinel-2 (remoto)", bands=[3, 2, 1], rescale="1000,3500", zoom_to_layer=False)
mapa

<figure style="text-align: center;">
  <img src="https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/contenidos/iv-procesamiento-datos-geoespaciales/img/leafmap-12-cog.png" alt="La imagen Sentinel-2 servida desde el repositorio del curso mediante titiler" width="900">
  <figcaption><strong>Figura 12</strong>. Un raster remoto agregado con add_cog_layer(): las teselas las produce un servicio en la nube a partir del archivo publicado. Fuente: elaboración propia.</figcaption>
</figure>

Para que eso funcione bien, el archivo debe ser un COG. `leafmap.cog_validate()` dice si lo es y por qué no; `leafmap.image_to_cog()` lo convierte, agregando los bloques y las vistas previas. La imagen del curso no lo es (se escribió con rasterio sin esas opciones), y la conversión la deja lista para publicarse:

In [17]:
# Validación y conversión a GeoTIFF optimizado para la nube (COG)
valido, errores, advertencias = leafmap.cog_validate(GANDOCA)
print("¿Es COG?", valido)
for error in errores:
    print("  -", error)

leafmap.image_to_cog(GANDOCA, "gandoca-cog.tif")
print("Convertido: ¿es COG?", leafmap.cog_validate("gandoca-cog.tif")[0], f"| {os.path.getsize('gandoca-cog.tif') / 1e6:.1f} MB")

The following warnings were found:


- The file is greater than 512xH or 512xW, it is recommended to include internal overviews


The following errors were found:


- The file is greater than 512xH or 512xW, but is not tiled


- The offset of the main IFD should be < 300. It is 11353386 instead


...

Adding overviews...


Updating dataset tags...


Writing output to: /home/mfvargas/gf0657-programacionsig/2026-ii/git/2026-ii/contenidos/iv-procesamiento-datos-geoespaciales/gandoca-cog.tif


¿Es COG? False
  - The file is greater than 512xH or 512xW, but is not tiled
  - The offset of the main IFD should be < 300. It is 11353386 instead
  - The offset of the first block of the image should be after its IFD


Convertido: ¿es COG? True | 16.5 MB


*Ejercicios de esta sección: 6, 7 y 8, en la sección de ejercicios al final del cuaderno.*

## Interacción: dibujar y consultar

Lo que distingue a un *widget* es el camino de regreso: lo que se dibuja en el mapa llega a Python. Con las herramientas de dibujo se traza un polígono, un rectángulo o un marcador, y el mapa lo guarda en `user_roi` (la última figura, como GeoJSON) y `user_rois` (todas); `user_roi_bounds()` retorna sus límites y `save_draw_features()` las escribe en un archivo. Es la forma de definir un **área de interés** a mano, sin coordenadas, y usarla después en el análisis: por ejemplo, con `leafmap.clip_image()`, que recorta un raster con ella. Dibuje un polígono sobre el mapa siguiente y luego ejecute la celda que le sigue:

In [18]:
# Dibuje un polígono o rectángulo sobre este mapa con las herramientas de la izquierda
mapa = leafmap.Map(center=(9.59, -82.63), zoom=13, height="500px")
mapa.add_raster(GANDOCA, indexes=[3, 2, 1], vmin=1000, vmax=3500, layer_name="Sentinel-2", zoom_to_layer=False)
mapa

In [19]:
# El dibujo vuelve a Python como GeoJSON; con él se recorta el raster
if mapa.user_roi is None:
    print("Aún no hay nada dibujado: trace un polígono sobre el mapa de la celda anterior y vuelva a ejecutar esta celda.")
else:
    print("Límites del área dibujada:", mapa.user_roi_bounds())
    mapa.save_draw_features("area-interes.geojson")
    area = gpd.read_file("area-interes.geojson")
    print("Área:", round(area.to_crs(8908).area.sum() / 1e6, 2), "km²")

    leafmap.clip_image(GANDOCA, mapa.user_roi, "gandoca-recorte.tif")
    print("Recorte escrito en gandoca-recorte.tif")

Aún no hay nada dibujado: trace un polígono sobre el mapa de la celda anterior y vuelva a ejecutar esta celda.


El **inspector** de la barra de herramientas (ícono de la lupa sobre el mapa) hace la consulta inversa: al hacer clic en el mapa muestra el valor de cada capa raster en ese punto y los atributos de las capas vectoriales, lo que en el cuaderno de rasterio se hizo con `sample()`.

*Ejercicios de esta sección: 9, en la sección de ejercicios al final del cuaderno.*

## Funciones de análisis y conversión

Además de la clase `Map`, el módulo `leafmap` reúne funciones de utilidad que no dibujan nada, pero que resuelven tareas frecuentes del flujo de trabajo geoespacial en una línea. Las de esta tabla son las más útiles para el curso; `leafmap.zonal_stats()` (estadísticas zonales, el tema del cuaderno de rasterio) requiere instalar además la biblioteca rasterstats.

<figure style="text-align: center; margin: 20px 0;">
    <figcaption><strong>Tabla 1</strong>. Funciones de utilidad del módulo leafmap. Fuente: elaboración propia con base en Wu (s. f.).</figcaption>
    <table class="table table-bordered table-striped" style="margin: 0 auto;">
    <thead>
      <tr><th>Función</th><th>Qué hace</th></tr>
    </thead>
    <tbody>
      <tr><td><code>download_file(url, archivo)</code></td><td>Descarga un archivo de una URL (y descomprime los ZIP)</td></tr>
      <tr><td><code>csv_to_gdf(csv, latitude, longitude)</code></td><td>Convierte una tabla con coordenadas en un GeoDataFrame</td></tr>
      <tr><td><code>cog_validate(archivo)</code>, <code>image_to_cog(origen, destino)</code></td><td>Valida y crea GeoTIFF optimizados para la nube</td></tr>
      <tr><td><code>cog_info(url)</code>, <code>cog_bounds(url)</code></td><td>Metadatos y límites de un raster remoto, sin descargarlo</td></tr>
      <tr><td><code>clip_image(raster, mascara, destino)</code></td><td>Recorta un raster con un polígono (dibujado o de un archivo)</td></tr>
      <tr><td><code>zonal_stats(vectores, raster)</code></td><td>Estadísticas zonales (requiere rasterstats)</td></tr>
      <tr><td><code>stac_search()</code>, <code>stac_bands()</code></td><td>Búsqueda de imágenes en catálogos STAC, como el Planetary Computer</td></tr>
      <tr><td><code>create_timelapse()</code>, <code>vector_to_gif()</code></td><td>Animaciones a partir de series de imágenes o de capas</td></tr>
    </tbody>
    </table>
</figure>

Dos ejemplos. Las ciudades de Natural Earth, que en el cuaderno de geopandas I se convirtieron en puntos con `points_from_xy()`, en una línea; y los metadatos de la imagen de Gandoca leídos de su URL, sin descargarla:

In [20]:
# Tabla con coordenadas a GeoDataFrame, y metadatos de un raster remoto
ciudades = leafmap.csv_to_gdf(f"{DATOS_NE}/ciudades.csv", latitude="latitud", longitude="longitud")
print(type(ciudades).__name__, ciudades.shape, ciudades.crs)

info = leafmap.cog_info(f"{DATOS_S2}/gandoca-20240204.tif")
print("SRC:", info["crs"], "| bandas:", info["count"], "| tipo:", info["dtype"], "| tamaño:", info["width"], "×", info["height"])
print("Límites en WGS84:", [round(v, 3) for v in leafmap.cog_bounds(f"{DATOS_S2}/gandoca-20240204.tif")])

GeoDataFrame (243, 8) EPSG:4326
SRC: http://www.opengis.net/def/crs/EPSG/0/32617 | bandas: 4 | tipo: uint16 | tamaño: 1811 × 1231


Límites en WGS84: [-82.712, 9.534, -82.547, 9.646]


## Exportar y publicar

`to_html()` escribe el mapa como una página HTML con el estado de los *widgets* incrustado: muestra las capas vectoriales y los mapas base, pero no las funciones interactivas ni los rasters locales (que dependen del servidor de teselas de la sesión). Para un mapa que deba publicarse, el camino es `leafmap.foliumap`: la misma clase `Map` y los mismos métodos (`add_basemap()`, `add_gdf()`, `add_data()`, `add_points_from_xy()`, `add_heatmap()`, `split_map()` con mapas base…) sobre folium, con un resultado que se guarda con `to_html()` y se publica como en el cuaderno anterior. Y en la sección V, `to_streamlit()` incrusta cualquiera de los dos en una aplicación web:

In [21]:
# El mismo mapa con el motor de folium, listo para publicarse como HTML
import leafmap.foliumap as foliumap

mapa_folium = foliumap.Map(center=CENTRO_CR, zoom=8, height="500px")
mapa_folium.add_basemap("Esri.WorldGrayCanvas")
mapa_folium.add_data(cantones, column="densidad", scheme="Quantiles", k=5, cmap="YlOrRd",
                     legend_title="Densidad de población, 2022 (hab/km²)", layer_name="Densidad de población",
                     style={"color": "white", "weight": 0.7, "fillOpacity": 0.8})
mapa_folium.add_circle_markers_from_xy(aerodromos.drop(columns="geometry"), x="longitud", y="latitud",
                                       radius=3, color=AZUL, fill_color=AZUL, fill_opacity=0.8, popup=["nombre"], layer_name="Aeródromos")
mapa_folium.add_layer_control()

mapa_folium.to_html("cantones-leafmap.html")
print(f"cantones-leafmap.html: {os.path.getsize('cantones-leafmap.html') / 1e6:.2f} MB")
mapa_folium

cantones-leafmap.html: 0.81 MB


Este último mapa sí aparece interactivo en el sitio web del curso, porque es un mapa de folium. La elección entre los dos motores es, entonces, de propósito: ipyleaflet para explorar, dibujar y consultar mientras se analiza; folium para publicar el resultado.

*Ejercicios de esta sección: 10, en la sección de ejercicios al final del cuaderno.*

## Clínica de errores

Los errores propios de leafmap, provocados a propósito. El primero no detiene el programa; los otros tres lanzan excepciones claras.

In [22]:
# Error silencioso: un nombre de mapa base que no existe
# add_basemap() imprime la lista completa de nombres válidos (más de 150 líneas) y no agrega nada:
# el mapa sigue igual y el programa continúa. El nombre correcto es "Esri.WorldImagery".
mapa = leafmap.Map(center=CENTRO_CR, zoom=8, height="300px")
mapa.add_basemap("Esri.Imagery")
print("Capas del mapa:", [capa.name for capa in mapa.layers])

Basemap can only be one of the following:
  OpenStreetMap
  Esri.WorldStreetMap
  Esri.WorldImagery
  Esri.WorldTopoMap
  FWS NWI Wetlands
  FWS NWI Wetlands Raster
  NLCD 2021 CONUS Land Cover
  NLCD 2019 CONUS Land Cover
  NLCD 2016 CONUS Land Cover
  NLCD 2013 CONUS Land Cover
  NLCD 2011 CONUS Land Cover
  NLCD 2008 CONUS Land Cover
  NLCD 2006 CONUS Land Cover
  NLCD 2004 CONUS Land Cover
  NLCD 2001 CONUS Land Cover
  USGS NAIP Imagery
  USGS NAIP Imagery False Color
  USGS NAIP Imagery NDVI
  USGS Hydrography
  USGS 3DEP Elevation
  USGS 3DEP Elevation Index
  ESA Worldcover 2020
  ESA Worldcover 2020 S2 FCC
  ESA Worldcover 2020 S2 TCC
  ESA Worldcover 2021
  ESA Worldcover 2021 S2 FCC
  ESA Worldcover 2021 S2 TCC
  USGS.Imagery
  BaseMapDE.Color
  BaseMapDE.Grey
  BasemapAT.basemap
  BasemapAT.grau
  BasemapAT.highdpi
  BasemapAT.orthofoto
  BasemapAT.overlay
  BasemapAT.surface
  BasemapAT.terrain
  CartoDB.DarkMatter
  CartoDB.DarkMatterNoLabels
  CartoDB.DarkMatterOnlyLabel

In [23]:
# KeyError: 'lon'
# add_points_from_xy() y add_circle_markers_from_xy() esperan los nombres de las columnas de coordenadas
# en x e y; por defecto son "longitude" y "latitude". Aquí las columnas se llaman longitud y latitud.
mapa = leafmap.Map(center=CENTRO_CR, zoom=8, height="300px")
mapa.add_points_from_xy(aerodromos.drop(columns="geometry"), x="lon", y="lat")

KeyError: 'lon'

In [24]:
# OSError: "Path does not exist"
# add_raster() necesita un archivo local existente (o una URL válida): el nombre está mal escrito.
mapa = leafmap.Map(center=CENTRO_CR, zoom=8, height="300px")
mapa.add_raster("altitud-centroamerica.tiff", layer_name="Altitud")

OSError: Path does not exist: /home/mfvargas/gf0657-programacionsig/2026-ii/git/2026-ii/contenidos/iv-procesamiento-datos-geoespaciales/altitud-centroamerica.tiff

In [25]:
# ValueError: el esquema de clasificación no existe
# scheme usa los nombres de mapclassify, con mayúsculas: "Quantiles", "NaturalBreaks", "EqualInterval"...
mapa = leafmap.Map(center=CENTRO_CR, zoom=8, height="300px")
mapa.add_data(cantones, column="densidad", scheme="quintiles", k=5, cmap="YlOrRd", layer_name="Densidad")

ValueError: quintiles is not a valid scheme. It must be one of ['BoxPlot', 'EqualInterval', 'FisherJenks', 'FisherJenksSampled', 'HeadTailBreaks', 'JenksCaspall', 'JenksCaspallForced', 'JenksCaspallSampled', 'MaxP', 'MaximumBreaks', 'NaturalBreaks', 'Quantiles', 'Percentiles', 'StdMean', 'UserDefined'].

Un error más, que no se puede provocar aquí: en Google Colab, si no se ejecuta `output.enable_custom_widget_manager()` antes de crear el mapa, la celda no muestra nada (o muestra un mensaje de *widget* no disponible), sin excepción. La primera celda del cuaderno lo resuelve.

## Asistentes de IA para mapear con leafmap

Las pautas de los cuadernos anteriores aplican, con una particularidad: leafmap cambia con frecuencia (varias versiones al año) y los asistentes mezclan versiones, motores (`leafmap`, `leafmap.foliumap`, `leafmap.maplibregl`) y nombres de argumentos. Las comprobaciones son cuatro: que los **métodos y argumentos existan** en la versión instalada (`help(leafmap.Map.add_data)` o la referencia de la API lo dicen en segundos); que el **motor** sea el adecuado al propósito (ipyleaflet para explorar, folium para publicar); que las **columnas de coordenadas** y los nombres de capa coincidan con los datos; y que los **rasters** tengan `vmin`, `vmax` y `nodata` razonables, porque sin ellos el mapa sale negro o lavado. Un prompt de ejemplo:

> Actúe como asistente de visualización geoespacial con leafmap, versión 0.63, con el motor de ipyleaflet **(rol)**. Tengo un GeoDataFrame `cantones` con 84 polígonos en WGS84 y las columnas `canton`, `provincia` y `poblacion` (entero, con dos valores faltantes), y un GeoTIFF local `altitud-centroamerica.tif` de una banda, en WGS84, con valores de 0 a 3 800 m y −32 768 como valor sin dato **(contexto)**. Escriba el código de un mapa centrado en Costa Rica con el mapa base de imágenes de Esri, la altitud como capa raster con la paleta `terrain` entre 0 y 3 500 m al 70 % de opacidad, los cantones como coropletas de la población en cinco clases por cortes naturales con leyenda, y una barra de color para la altitud **(tarea)**. Use `leafmap.Map()`, `add_basemap()`, `add_raster()`, `add_data()` y `add_colorbar()`, nombre cada capa, y agregue un comentario por paso **(formato)**.

## Resumen

- **leafmap** construye mapas interactivos sobre **ipyleaflet**, que hace del mapa un *widget* de Jupyter: lo que se dibuja o consulta en el mapa vuelve a Python. Trae barra de herramientas, más de 150 **mapas base** (`add_basemap()`), herramientas de dibujo, inspector y medición; en el sitio web del curso sus mapas aparecen como capturas, en Colab o VS Code son interactivos.
- `leafmap.Map(center=(lat, lon), zoom, height)` crea el mapa. Las capas vectoriales se agregan en una línea: `add_gdf()` (estilo, resaltado, `info_mode`), `add_data()` (coropletas clasificadas con leyenda), `add_points_from_xy()` (marcadores agrupados), `add_circle_markers_from_xy()`, `add_heatmap()` y `add_wms_layer()`. leafmap reproyecta a WGS84 por su cuenta.
- `add_raster()` muestra un GeoTIFF local como teselas mediante **localtileserver**, con `colormap`, `vmin`, `vmax`, `nodata`, `opacity` e `indexes` para las bandas; `split_map()` compara dos capas con un control deslizante; `add_cog_layer()` muestra un raster remoto por **titiler**, y `cog_validate()` e `image_to_cog()` preparan **GeoTIFF optimizados para la nube**.
- Lo dibujado queda en `user_roi` y `user_rois`, se guarda con `save_draw_features()` y sirve de máscara para `clip_image()`; el inspector consulta valores al hacer clic.
- Las funciones del módulo (`download_file()`, `csv_to_gdf()`, `cog_info()`, `zonal_stats()`, búsqueda STAC) resuelven tareas del flujo de trabajo sin dibujar.
- `to_html()` exporta el estado del mapa; para publicar, el motor `leafmap.foliumap` ofrece los mismos métodos sobre folium, y `to_streamlit()` lleva cualquiera de los dos a una aplicación web.

## Ejercicios

Los ejercicios se agrupan según la sección del cuaderno a la que corresponden; se recomienda realizarlos al concluir la sección respectiva. Resuélvalos en este mismo cuaderno (en su copia de Colab o local), después de ejecutar las celdas de las secciones anteriores. Varios incluyen una **celda de verificación** que examina el mapa creado (sus capas, por nombre y tipo, y sus controles): para que funcione, nombre los objetos y las capas como se indica en el enunciado y ejecute la celda tal cual después de resolver el ejercicio (en la versión publicada muestra un recordatorio, porque los ejercicios no están resueltos). Las [soluciones de estos ejercicios](https://gf0657-programacionsig.github.io/2026-ii/soluciones-leafmap-mapeo-interactivo) se publican después de la clase correspondiente.

In [26]:
# Función auxiliar de las celdas de verificación: nombres de las capas de un mapa
def capas(mapa):
    return [capa.name for capa in mapa.layers]

### Un primer mapa

1. Cree en `mapa_1` un mapa centrado en el cantón donde vive o estudia, con el acercamiento adecuado para verlo completo, la barra de escala, la herramienta de medición y los mapas base `"OpenTopoMap"` y `"Esri.WorldImagery"` además del predeterminado. Mida con la herramienta de medición la distancia entre dos lugares que conozca y anótela en una celda de texto.

In [27]:
# Celda de verificación del ejercicio 1: ejecútela tal cual
try:
    assert isinstance(mapa_1, leafmap.Map), "mapa_1 debe ser un leafmap.Map"
    assert {"OpenTopoMap", "Esri.WorldImagery"} <= set(capas(mapa_1)), "mapa_1 debe tener los mapas base OpenTopoMap y Esri.WorldImagery"
    print("Ejercicio 1: ¡correcto!")
except NameError:
    print("Aún no se ha definido mapa_1 (ejercicio 1).")

Aún no se ha definido mapa_1 (ejercicio 1).


### Capas vectoriales

2. Cree en `mapa_2` un mapa con las provincias (`add_gdf()`) en una capa llamada `"Provincias"`, con relleno de color, borde blanco, resaltado al pasar el cursor y los atributos al hacer clic (`info_mode="on_click"`), sobre el fondo gris de Esri. ¿En qué se diferencia el recuadro de atributos de la etiqueta emergente de folium?

In [28]:
# Celda de verificación del ejercicio 2: ejecútela tal cual
try:
    assert isinstance(mapa_2, leafmap.Map) and "Provincias" in capas(mapa_2), "mapa_2 debe tener una capa llamada Provincias"
    print("Ejercicio 2: ¡correcto!")
except NameError:
    print("Aún no se ha definido mapa_2 (ejercicio 2).")

Aún no se ha definido mapa_2 (ejercicio 2).


3. Cree en `mapa_3` un mapa de coropletas de la **población** de los cantones con `add_data()`, clasificada en cinco clases por cortes naturales (`"NaturalBreaks"`), con la paleta `"PuBu"`, la leyenda con título y año, y la capa llamada `"Población"`. Compare los intervalos de la leyenda con los quintiles del cuaderno de folium: ¿qué cambia con los cortes naturales y cuándo preferiría cada esquema?

In [29]:
# Celda de verificación del ejercicio 3: ejecútela tal cual
try:
    assert isinstance(mapa_3, leafmap.Map) and "Población" in capas(mapa_3), "mapa_3 debe tener una capa llamada Población"
    print("Ejercicio 3: ¡correcto!")
except NameError:
    print("Aún no se ha definido mapa_3 (ejercicio 3).")

Aún no se ha definido mapa_3 (ejercicio 3).


4. Con el DataFrame de los cuatro aeropuertos internacionales del cuaderno de folium (nombre, ciudad, longitud y latitud), cree en `mapa_4` un mapa con los aeródromos como círculos pequeños (`add_circle_markers_from_xy()`, capa `"Aeródromos"`) y los aeropuertos como marcadores (`add_points_from_xy()`, capa `"Aeropuertos internacionales"`, con el nombre y la ciudad en la ventana emergente). ¿Qué pasa con los marcadores agrupados al acercarse al valle Central?

In [30]:
# Celda de verificación del ejercicio 4: ejecútela tal cual
try:
    assert isinstance(mapa_4, leafmap.Map), "mapa_4 debe ser un leafmap.Map"
    assert {"Aeródromos", "Aeropuertos internacionales"} <= set(capas(mapa_4)), "mapa_4 debe tener las capas Aeródromos y Aeropuertos internacionales"
    print("Ejercicio 4: ¡correcto!")
except NameError:
    print("Aún no se ha definido mapa_4 (ejercicio 4).")

Aún no se ha definido mapa_4 (ejercicio 4).


5. Cree en `mapa_5` un mapa centrado en el lago Arenal (10,49° N, 84,85° O) con el acercamiento 12, el mapa base topográfico y, del WMS del IGN a 1:200 000, las capas `lago_o_laguna_200k`, `reddrenaje_200k` y `curvas_de_nivel_200k`. Abra el inspector desde la barra de herramientas y haga clic sobre el lago: ¿qué muestra para una capa WMS y por qué?

In [31]:
# Celda de verificación del ejercicio 5: ejecútela tal cual
try:
    assert isinstance(mapa_5, leafmap.Map), "mapa_5 debe ser un leafmap.Map"
    assert sum(isinstance(capa, ipyleaflet.WMSLayer) for capa in mapa_5.layers) >= 3, "mapa_5 debe tener al menos tres capas WMS"
    print("Ejercicio 5: ¡correcto!")
except NameError:
    print("Aún no se ha definido mapa_5 (ejercicio 5).")

Aún no se ha definido mapa_5 (ejercicio 5).


### Capas raster

6. Descargue con `leafmap.download_file()` la temperatura media anual mundial de WorldClim (`DATOS_WC + "/temperatura.tif"`, una banda `float32` con −3,4 × 10³⁸ como valor sin dato) y créela en `mapa_6` como capa raster llamada `"Temperatura"`, con la paleta divergente `"RdBu_r"` entre −10 y 30 °C, el valor sin dato declarado con `nodata`, opacidad 0,8 y una barra de color con unidades, en un mapa del mundo (centro en (10, 0), acercamiento 2). ¿Dónde están las celdas que saturan en cada extremo de la paleta?

In [32]:
# Celda de verificación del ejercicio 6: ejecútela tal cual
try:
    assert isinstance(mapa_6, leafmap.Map) and "Temperatura" in capas(mapa_6), "mapa_6 debe tener una capa llamada Temperatura"
    assert "api/tiles" in mapa_6.find_layer("Temperatura").url, "La capa Temperatura debe servirla localtileserver (add_raster)"
    print("Ejercicio 6: ¡correcto!")
except NameError:
    print("Aún no se ha definido mapa_6 (ejercicio 6).")

Aún no se ha definido mapa_6 (ejercicio 6).


7. Cree en `mapa_7` un mapa dividido (`split_map()`) centrado en Gandoca-Manzanillo que compare las imágenes de satélite de Esri (a la izquierda) con la imagen Sentinel-2 en color verdadero (a la derecha, con el estiramiento del cuaderno). ¿Son de la misma fecha? ¿Qué diferencias de resolución y de nubosidad observa?

In [33]:
# Celda de verificación del ejercicio 7: ejecútela tal cual
try:
    assert isinstance(mapa_7, leafmap.Map), "mapa_7 debe ser un leafmap.Map"
    assert any(isinstance(control, ipyleaflet.SplitMapControl) for control in mapa_7.controls), "mapa_7 debe tener el control de mapa dividido"
    print("Ejercicio 7: ¡correcto!")
except NameError:
    print("Aún no se ha definido mapa_7 (ejercicio 7).")

Aún no se ha definido mapa_7 (ejercicio 7).


8. Convierta la altitud de Centroamérica (`ALTITUD`) a GeoTIFF optimizado para la nube en `"altitud-cog.tif"` con `leafmap.image_to_cog()`, guarde en `valido_8` el primer elemento del resultado de `leafmap.cog_validate()` sobre el archivo nuevo, y compare los tamaños de los dos archivos. Luego, en `mapa_8`, muestre la versión COG con `add_raster()` y la paleta `"YlGn"` entre 0 y 2 000 m, en una capa llamada `"Altitud COG"`. ¿Qué cambió en el archivo para que sea más rápido de consultar, y por qué el tamaño puede subir o bajar?

In [34]:
# Celda de verificación del ejercicio 8: ejecútela tal cual
try:
    assert valido_8 is True, "valido_8 debe ser True: el archivo convertido debe ser un COG válido"
    assert os.path.exists("altitud-cog.tif"), "Debe existir el archivo altitud-cog.tif"
    assert isinstance(mapa_8, leafmap.Map) and "Altitud COG" in capas(mapa_8), "mapa_8 debe tener una capa llamada Altitud COG"
    print("Ejercicio 8: ¡correcto!")
except NameError:
    print("Aún no se ha definido valido_8 o mapa_8 (ejercicio 8).")

Aún no se ha definido valido_8 o mapa_8 (ejercicio 8).


### Interacción

9. Cree en `mapa_9` un mapa con la imagen Sentinel-2 de Gandoca y dibuje sobre él, con la herramienta de polígono, el contorno aproximado de la laguna de Gandoca (al sureste de la imagen, junto a la costa). Guarde el dibujo en `"laguna.geojson"` con `save_draw_features()`, léalo con geopandas en `laguna_9`, calcule su área en hectáreas en CRTM05 y recorte la imagen con `leafmap.clip_image()` en `"laguna.tif"`. Abra el recorte en QGIS o con rasterio y compruebe sus dimensiones.

In [35]:
# Celda de verificación del ejercicio 9: ejecútela tal cual
try:
    assert isinstance(laguna_9, gpd.GeoDataFrame) and len(laguna_9) >= 1, "laguna_9 debe ser un GeoDataFrame con al menos un polígono"
    assert laguna_9.geom_type.iloc[0] in ("Polygon", "MultiPolygon"), "El dibujo debe ser un polígono"
    assert os.path.exists("laguna.tif"), "Debe existir el recorte laguna.tif"
    print("Ejercicio 9: ¡correcto!")
except NameError:
    print("Aún no se ha definido laguna_9 (ejercicio 9).")

Aún no se ha definido laguna_9 (ejercicio 9).


### Exportar, predicción, asistentes y tarea 3

10. Reconstruya el mapa de coropletas de la población del ejercicio 3 con el motor de folium (`leafmap.foliumap`) en `mapa_10`, agréguele los aeropuertos internacionales como marcadores, el control de capas y expórtelo con `to_html()` a `"poblacion-leafmap.html"` en `ruta_10`. Ábralo en el navegador y compárelo con el mapa equivalente del cuaderno de folium: ¿qué controles tiene cada uno?

In [36]:
# Celda de verificación del ejercicio 10: ejecútela tal cual
try:
    import folium
    assert isinstance(mapa_10, folium.Map), "mapa_10 debe ser un mapa del motor de folium (leafmap.foliumap.Map)"
    assert os.path.exists(ruta_10) and ruta_10.endswith(".html"), "ruta_10 debe ser la ruta de un archivo HTML existente"
    print("Ejercicio 10: ¡correcto!")
except NameError:
    print("Aún no se ha definido mapa_10 o ruta_10 (ejercicio 10).")

Aún no se ha definido mapa_10 o ruta_10 (ejercicio 10).


11. **Prediga antes de ejecutar**: ¿qué muestra cada mapa? ¿En cuál de los dos se ve el cantón y por qué? Escriba su predicción en una celda de texto y luego ejecute el código (en dos celdas, una por mapa).

```python
san_jose_crtm05 = gpd.read_file(f"{DATOS_IGN}/cantones.gpkg").query("canton == 'San José'")
print(san_jose_crtm05.crs)

import folium
mapa_a = folium.Map(location=[9.93, -84.08], zoom_start=12)
folium.GeoJson(san_jose_crtm05).add_to(mapa_a)
mapa_a
```

```python
mapa_b = leafmap.Map(center=(9.93, -84.08), zoom=12, height="400px")
mapa_b.add_gdf(san_jose_crtm05, layer_name="San José", zoom_to_layer=False)
mapa_b
```

<details>
<summary>Explicación (ábrala después de predecir)</summary>

La capa está en CRTM05 (EPSG:8908), con coordenadas en metros. folium la pasa a GeoJSON tal cual y Leaflet no puede ubicar coordenadas de cientos de miles de "grados": el primer mapa queda vacío, sin error (el error silencioso 2 del cuaderno de folium). leafmap, en cambio, reproyecta a WGS84 antes de convertir a GeoJSON, y el segundo mapa muestra el cantón. La comodidad tiene un precio: conviene saber que ocurre, porque la reproyección automática no aplica a todo (las columnas de coordenadas de una tabla, por ejemplo, deben estar en WGS84).

</details>

12. Pida a un asistente de IA el código del prompt de ejemplo de la sección de asistentes (altitud y coropletas de la población) y verifíquelo con las cuatro comprobaciones de esa sección: métodos y argumentos existentes en la versión instalada, motor adecuado, columnas y nombres de capa correctos, y rango y valor sin dato del raster. Documente en una celda de texto el asistente, el prompt, el código y el resultado de las comprobaciones.

13. Para la tarea 3, repita con leafmap el mapa de la capa de su tema que hizo en el cuaderno de folium, y agréguele lo que folium no ofrecía: un raster relacionado con su tema (el del ejercicio 15 del cuaderno de rasterio, o uno de WorldClim) con paleta y rango adecuados, y una comparación con `split_map()` entre el raster y las imágenes de satélite. Anote en una celda de texto qué le aporta el raster a las preguntas de su tema y qué motor (ipyleaflet o folium) usaría para publicar el resultado.

## Referencias bibliográficas

Development Seed. (s. f.). *TiTiler: A modern dynamic tile server built on top of FastAPI and Rasterio/GDAL* [Software]. Recuperado el 25 de octubre de 2026, de https://developmentseed.org/titiler/
\
\
Instituto Geográfico Nacional. (s. f.). *Sistema Nacional de Información Territorial (SNIT)* [Servicios WMS y WFS]. Registro Nacional de Costa Rica. Recuperado el 25 de octubre de 2026, de https://www.snitcr.go.cr/
\
\
Instituto Nacional de Estadística y Censos. (2023). *Resultados de la Estimación de Población y Vivienda 2022* [Conjunto de datos]. https://inec.cr/
\
\
Jupyter Widgets contributors. (s. f.). *ipyleaflet: Interactive maps in the Jupyter notebook* [Software]. Recuperado el 25 de octubre de 2026, de https://ipyleaflet.readthedocs.io/
\
\
Microsoft. (s. f.). *Sentinel-2 Level-2A*. Microsoft Planetary Computer. Recuperado el 18 de octubre de 2026, de https://planetarycomputer.microsoft.com/dataset/sentinel-2-l2a
\
\
Natural Earth. (s. f.). *Natural Earth: Free vector and raster map data* [Conjunto de datos]. Recuperado el 4 de octubre de 2026, de https://www.naturalearthdata.com/
\
\
Sullivan, B. (s. f.). *localtileserver: A Python package for serving tiles from large raster files* [Software]. Recuperado el 25 de octubre de 2026, de https://localtileserver.banesullivan.com/
\
\
WorldClim. (s. f.). *WorldClim version 2.1* [Conjunto de datos]. Recuperado el 18 de octubre de 2026, de https://www.worldclim.org/data/worldclim21.html
\
\
Wu, Q. (2021). Leafmap: A Python package for interactive mapping and geospatial analysis with minimal coding in a Jupyter environment. *Journal of Open Source Software, 6*(63), 3414. [https://joss.theoj.org/papers/10.21105/joss.03414](https://joss.theoj.org/papers/10.21105%2Fjoss.03414)
\
\
Wu, Q. (s. f.). *leafmap documentation* (versión 0.63). Recuperado el 25 de octubre de 2026, de https://leafmap.org/